## Project Aim

The primary objective of this study is to explore employee characteristics and discover hidden patterns within an organization's workforce through unsupervised machine learning techniques. By analyzing various factors, including employee demographics, job characteristics, job satisfaction, and career history, this project aims to identify distinct groups of employees with similar characteristics and working conditions.

To achieve this objective, clustering algorithms such as K-Means, DBSCAN, and Hierarchical Clustering are implemented to segment employees into meaningful groups without relying on predefined target labels. The performance of these models is evaluated using appropriate clustering metrics, including the Silhouette Score, Davies-Bouldin Index, and Calinski-Harabasz Index, to assess the quality and separation of the resulting clusters.

Furthermore, employee attrition is examined as an additional variable to investigate whether different employee groups exhibit different turnover patterns. Based on the clustering results, the study seeks to identify meaningful workforce segments and provide data-driven insights and strategic recommendations to help the Human Resources department better understand employee needs, improve retention strategies, and support more effective workforce management.

## Importing Libraries

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import chardet

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

In [28]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "archive"

df_general = pd.read_csv(DATA_DIR / "general_data.csv")
df_employee = pd.read_csv(DATA_DIR / "employee_survey_data.csv")
df_manager = pd.read_csv(DATA_DIR / "manager_survey_data.csv")
df_inTime = pd.read_csv(DATA_DIR / "in_time.csv")
df_outTime = pd.read_csv(DATA_DIR / "out_time.csv")


## Initial Data inspection

### Encoding check

In [30]:
file_paths = {
    "General": DATA_DIR / "general_data.csv",
    "Employee": DATA_DIR / "employee_survey_data.csv",
    "Manager": DATA_DIR / "manager_survey_data.csv",
    "In-time": DATA_DIR / "in_time.csv",
    "Out-time": DATA_DIR / "out_time.csv"
}

for name, path in file_paths.items():
    try:
        pd.read_csv(path, encoding="utf-8")
        print(f"{name}: UTF-8")
    except UnicodeDecodeError:
        print(f"{name}: Not UTF-8")

General: UTF-8
Employee: UTF-8
Manager: UTF-8
In-time: UTF-8
Out-time: UTF-8


### Reading dataset

In [25]:
df_general.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeID,Gender,...,NumCompaniesWorked,Over18,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,...,1.0,Y,11,8,0,1.0,6,1,0,0
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,...,0.0,Y,23,8,1,6.0,3,5,1,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,...,1.0,Y,15,8,3,5.0,2,5,0,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,...,3.0,Y,11,8,3,13.0,5,8,7,5
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,...,4.0,Y,12,8,2,9.0,2,6,0,4


### Data inspection

In [7]:
# General
df_general.info()

df_general.describe(include='all')

df_general.isnull().sum()

df_general.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      4410 non-null   int64  
 1   Attrition                4410 non-null   str    
 2   BusinessTravel           4410 non-null   str    
 3   Department               4410 non-null   str    
 4   DistanceFromHome         4410 non-null   int64  
 5   Education                4410 non-null   int64  
 6   EducationField           4410 non-null   str    
 7   EmployeeCount            4410 non-null   int64  
 8   EmployeeID               4410 non-null   int64  
 9   Gender                   4410 non-null   str    
 10  JobLevel                 4410 non-null   int64  
 11  JobRole                  4410 non-null   str    
 12  MaritalStatus            4410 non-null   str    
 13  MonthlyIncome            4410 non-null   int64  
 14  NumCompaniesWorked       4391 non-n

np.int64(0)

In [8]:
# Employee
df_employee.info()

df_employee.describe(include='all')

df_employee.isnull().sum()

df_employee.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   EmployeeID               4410 non-null   int64  
 1   EnvironmentSatisfaction  4385 non-null   float64
 2   JobSatisfaction          4390 non-null   float64
 3   WorkLifeBalance          4372 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 137.9 KB


np.int64(0)

In [9]:
# Manager
df_manager.info()

df_manager.describe(include='all')

df_manager.isnull().sum()

df_manager.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   EmployeeID         4410 non-null   int64
 1   JobInvolvement     4410 non-null   int64
 2   PerformanceRating  4410 non-null   int64
dtypes: int64(3)
memory usage: 103.5 KB


np.int64(0)

In [23]:
# In-time
df_inTime.info()

df_inTime.describe(include='all')

missing_values = df_inTime.isnull().sum()

print("Total missing values:", missing_values.sum())

print("Number of duplicate rows:", df_inTime.duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Columns: 262 entries, Unnamed: 0 to 2015-12-31
dtypes: float64(12), int64(1), str(249)
memory usage: 8.8 MB
Total missing values: 109080
Number of duplicate rows: 0


In [24]:
# Out time
df_outTime.info()

df_outTime.describe(include='all')

missing_values = df_outTime.isnull().sum()
print("Total missing values:", missing_values.sum())

print("Number of duplicate rows:", df_outTime.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Columns: 262 entries, Unnamed: 0 to 2015-12-31
dtypes: float64(12), int64(1), str(249)
memory usage: 8.8 MB
Total missing values: 109080
Number of duplicate rows: 0


 ### Identify relationships

In [42]:
datasets = {
    "General": df_general,
    "Employee": df_employee,
    "Manager": df_manager,
    "In-time": df_inTime,
    "Out-time": df_outTime
}

dataset_names = list(datasets.keys())

for i in range(len(dataset_names)):
    for j in range(i + 1, len(dataset_names)):

        name1 = dataset_names[i]
        name2 = dataset_names[j]

        common_columns = set(datasets[name1].columns) & set(datasets[name2].columns)

        print(f"{name1} - {name2}: {common_columns}")

General - Employee: {'EmployeeID'}
General - Manager: {'EmployeeID'}
General - In-time: set()
General - Out-time: set()
Employee - Manager: {'EmployeeID'}
Employee - In-time: set()
Employee - Out-time: set()
Manager - In-time: set()
Manager - Out-time: set()
In-time - Out-time: {'2015-10-21', '2015-07-21', '2015-06-17', '2015-11-13', '2015-09-03', '2015-11-02', '2015-06-24', '2015-12-01', '2015-06-12', '2015-01-13', '2015-12-30', '2015-06-03', '2015-04-14', '2015-10-23', '2015-08-24', '2015-01-19', '2015-01-23', '2015-07-03', '2015-09-02', '2015-09-18', '2015-03-04', '2015-09-10', '2015-12-07', '2015-12-10', '2015-12-18', '2015-03-27', '2015-07-14', '2015-04-09', '2015-08-12', '2015-06-01', '2015-09-25', '2015-11-04', '2015-11-25', '2015-03-03', '2015-12-11', '2015-02-11', '2015-04-23', '2015-03-02', '2015-09-16', '2015-06-02', '2015-03-19', '2015-09-30', '2015-11-18', '2015-11-23', '2015-07-07', '2015-02-27', '2015-08-28', '2015-01-05', '2015-10-13', '2015-04-03', '2015-04-07', '2015-

## Data Loading and Integration

In [13]:
df_final = df_general.merge(df_employee, on='EmployeeID').merge(df_manager, on='EmployeeID')

In [14]:
df_final.head() # a dataframe containing complete information for each employee

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeID,Gender,...,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,...,1.0,6,1,0,0,3.0,4.0,2.0,3,3
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,...,6.0,3,5,1,4,3.0,2.0,4.0,2,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,...,5.0,2,5,0,3,2.0,2.0,1.0,3,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,...,13.0,5,8,7,5,4.0,4.0,3.0,2,3
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,...,9.0,2,6,0,4,4.0,1.0,3.0,3,3


In [15]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      4410 non-null   int64  
 1   Attrition                4410 non-null   str    
 2   BusinessTravel           4410 non-null   str    
 3   Department               4410 non-null   str    
 4   DistanceFromHome         4410 non-null   int64  
 5   Education                4410 non-null   int64  
 6   EducationField           4410 non-null   str    
 7   EmployeeCount            4410 non-null   int64  
 8   EmployeeID               4410 non-null   int64  
 9   Gender                   4410 non-null   str    
 10  JobLevel                 4410 non-null   int64  
 11  JobRole                  4410 non-null   str    
 12  MaritalStatus            4410 non-null   str    
 13  MonthlyIncome            4410 non-null   int64  
 14  NumCompaniesWorked       4391 non-n

## Data Cleaning

Xử lý giá trị thiếu (median/mode), loại bỏ dữ liệu trùng lặp, chuẩn hóa kiểu dữ liệu và tạo đặc trưng giờ làm việc từ dữ liệu in_time / out_time.

In [ ]:
# ============================================================
# DATA CLEANING: Missing values, Duplicates, Data types
# ============================================================

df_clean = df_final.copy()

# ---------- 1. XỬ LÝ GIÁ TRỊ THIẾU ----------
missing_before = df_clean.isnull().sum()
print('1. Giá trị thiếu TRƯỚC khi xử lý:')
print(missing_before[missing_before > 0])
print()

# Cột số đếm / số năm (lệch phải) -> điền bằng median
median_cols = ['NumCompaniesWorked', 'TotalWorkingYears']
for col in median_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Cột khảo sát thang 1-4 (thứ bậc) -> điền bằng mode để giữ giá trị hợp lệ
ordinal_cols = ['EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance']
for col in ordinal_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('   Giá trị thiếu SAU khi xử lý:', df_clean.isnull().sum().sum())
print()

# ---------- 2. XÓA DỮ LIỆU TRÙNG LẶP ----------
print('2. Dòng trùng hoàn toàn:', df_clean.duplicated().sum())
print('   EmployeeID bị trùng:', df_clean['EmployeeID'].duplicated().sum())

rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset='EmployeeID', keep='first')
print(f'   Số dòng: {rows_before} -> {len(df_clean)}')
print()

# ---------- 3. CHUẨN HÓA KIỂU DỮ LIỆU ----------
# Các cột float chỉ vì từng có NaN -> ép về int
int_cols = ['NumCompaniesWorked', 'TotalWorkingYears',
            'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance']
df_clean[int_cols] = df_clean[int_cols].astype(int)

# Cột văn bản -> category
cat_cols = ['Attrition', 'BusinessTravel', 'Department', 'EducationField',
            'Gender', 'JobRole', 'MaritalStatus']
df_clean[cat_cols] = df_clean[cat_cols].astype('category')

# Xóa cột hằng số (phương sai = 0, không có giá trị phân cụm)
const_cols = [c for c in df_clean.columns if df_clean[c].nunique() == 1]
print('3. Cột hằng số đã xóa:', const_cols)
print()
df_clean = df_clean.drop(columns=const_cols)

# ---------- 4. XỬ LÝ CỘT THỜI GIAN (in_time / out_time) ----------
def prepare_time_df(df):
    df = df.rename(columns={'Unnamed: 0': 'EmployeeID'}).set_index('EmployeeID')
    # Chuyển chuỗi -> datetime (NaT = nghỉ hoặc ngày lễ)
    return df.apply(pd.to_datetime, errors='coerce')

in_time = prepare_time_df(df_inTime)
out_time = prepare_time_df(df_outTime)

# Số giờ làm việc mỗi ngày
hours = (out_time - in_time).apply(lambda col: col.dt.total_seconds() / 3600)

# Ngày lễ công ty: ngày mà TẤT CẢ nhân viên đều không có dữ liệu
company_holidays = hours.columns[hours.isna().all()]
working_days = hours.drop(columns=company_holidays)

work_features = pd.DataFrame({
    'AvgWorkingHours': working_days.mean(axis=1),
    'DaysWorked': working_days.notna().sum(axis=1),
    'DaysAbsent': working_days.isna().sum(axis=1)
}).reset_index()

print(f'4. Ngày lễ công ty: {len(company_holidays)} | Ngày làm việc: {working_days.shape[1]}')

df_clean = df_clean.merge(work_features, on='EmployeeID', how='left')

# ---------- 5. KIỂM TRA CUỐI CÙNG ----------
print()
print('5. KIỂM TRA CUỐI CÙNG')
print('   Kích thước:', df_clean.shape)
print('   Giá trị thiếu:', df_clean.isnull().sum().sum())
print('   Dòng trùng:', df_clean.duplicated().sum())
print()
print('   Kiểu dữ liệu:')
print(df_clean.dtypes)
print()
print('   Thống kê đặc trưng giờ làm việc:')
print(df_clean[['AvgWorkingHours', 'DaysWorked', 'DaysAbsent']].describe())